# V8: all readers, unseen sentences
Question: can the frozen CNN classify new sentences when every test reader was represented during training? **No reader averaging and no averaging across words.** Each example remains a single reader/sentence's padded `(49,126,1)` word-level TRT_total array; TRT_total itself is a precomputed reading-window summary, not a waveform.
Use V7's exact fixed 66/22/22 sentence split and arbitrary mapping. Pool the 12 readers' separate examples: **792 training, 264 validation, 264 test**. No validation/test sentence identity occurs in training, for any reader. Validation/test sentences are also disjoint from each other.
One model per seed and target condition: seeds 45/62 x real/arbitrary labels = **4 fits**, roughly 10–20 minutes on a GPU (estimate). Same architecture, raw features, LR=0.0001 and stopping settings. Not leave-one-reader-out.
Each fit saves a verified ZIP containing all predictions, per-reader metrics, history and reusable inference weights. ZIPs are authoritative and summaries can be rebuilt without retraining. Expected Drive output about 10–15 MB; actual compression varies.
Limitations: one split and one mapping, only 22 unique test sentences; repeated readers/seeds are not independent new stimuli. Settings were selected in earlier experiments, so this is exploratory. Compared with V7, training increases from 10 to 12 readers and validation changes from one new reader to all seen readers. A difference is not a clean causal effect of reader familiarity alone. V7's missing prediction-file issue remains unresolved; V8 does not repair or validate it. No automatic extra experiments or p-value.

## 1. GPU setup
Choose a GPU runtime. Fixed source tag: `v8-seen-readers-v1`. No package installation or data download is performed; existing Drive inputs are reused.

In [ ]:
import os, sys, subprocess, json
import tensorflow as tf
assert tf.config.list_physical_devices('GPU'), 'Select a GPU runtime and rerun setup.'
from google.colab import drive
drive.mount('/content/drive')
repo = '/content/teco-v8-seen-readers'
run_ref = 'v8-seen-readers-v1'
if not os.path.isdir(repo):
    subprocess.run(['git', 'clone', '-q', '--branch', run_ref, '--single-branch',
                    'https://github.com/parmisbathayan/reproduce-teco-results.git', repo], check=True)
os.chdir(repo)
commit = subprocess.check_output(['git', 'rev-parse', 'HEAD'], text=True).strip()
assert commit == subprocess.check_output(['git', 'rev-parse', run_ref + '^{commit}'], text=True).strip()
assert not subprocess.check_output(['git','status','--porcelain'], text=True).strip(), 'Use a clean fixed revision.'
base = '/content/drive/MyDrive/Thesis/Results/reproduce_yaser_codes/eegonly'
out_dir = os.path.join(base, 'v8_seen_readers_unseen_sentences_45_62')
command = [sys.executable, '-u', '-B', 'src/v8_seen_readers.py',
           '--trt-dir', '/content/drive/MyDrive/Thesis/Data/TRT_Total',
           '--labels-csv', '/content/drive/MyDrive/Thesis/Data/teco_sentiment_labels_task1.csv',
           '--out-dir', out_dir, '--config', 'configs/v8_seen_readers_unseen_sentences.json',
           '--reference-out-dir', os.path.join(base, 'v4_low_lr_confirmation_45_62'),
           '--v7-out-dir', os.path.join(base, 'v7_unseen_sentences_45_62')]
def stream_run(arguments):
    with subprocess.Popen(arguments, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1) as process:
        try:
            for line in process.stdout:
                print(line, end='', flush=True)
            returncode = process.wait()
        except KeyboardInterrupt:
            process.terminate()
            try:
                process.wait(timeout=10)
            except subprocess.TimeoutExpired:
                process.kill()
                process.wait()
            print('Stopped. Verified job ZIPs remain saved; rerun setup, preflight and training to resume.')
            raise
    if returncode:
        raise subprocess.CalledProcessError(returncode, arguments)
print('GPU ready; fixed Git commit:', commit, flush=True)


## 2. Preflight — no training
Checks model, data and runtime against V4 and V7; requires the identical saved sentence split and label mapping; verifies sentence text/ID alignment across all readers and rejects exact nonzero-array overlap across sentence partitions. Existing V7 files are read only. Stop on any error; do not bypass checks.

In [ ]:
stream_run(command + ['--prepare-only'])
print('Expected: 792 training examples, 264 validation, 264 test; exactly balanced labels. No readers averaged.')


## 3. Run or resume four fits
Heartbeat every 30 seconds. Each fit saves one ZIP only after its predictions, metrics and reloaded weights verify. Resume trusts verified ZIPs, not just summary rows. Do not delete any local recovery directory mentioned by an error until the problem is investigated.

In [ ]:
stream_run(command)


## 4. Independently verify saved ZIPs and show results
No training here. Recomputes all per-reader metrics from each ZIP's predictions and rebuilds summaries, checking targets, class predictions and file checksums. Reports mean per-reader macro-F1, accuracy, correct counts and one-class output counts. Any V7 comparison remains provisional until its separate missing-file verification is resolved.

In [ ]:
stream_run(command + ['--verify-only'])
import pandas as pd
from IPython.display import display
with open(os.path.join(out_dir, 'completion.json')) as stream:
    completion = json.load(stream)
assert completion['complete'] and completion['completed_fits'] == 4
display(pd.read_csv(os.path.join(out_dir, 'summary_by_seed.csv')))
display(pd.read_csv(os.path.join(out_dir, 'summary_overall.csv')))
print('Four fitted models verified, all 1,056 test predictions retained in jobs/*.zip. Only 22 unique test sentence identities.')
print('No additional experiment is scheduled automatically. Send these results for review.')
